In [6]:
import os
import warnings 
import random 
from zipfile import ZipFile 
from pathlib import Path 
import copy

import pandas as pd 
import sklearn 
import numpy as np

import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 
from torchinfo import summary

import torch
from torch import nn 
from torch .utils.data import DataLoader, Dataset 
import torch.nn.functional as F
from torchvision import transforms 
from torchvision.transforms import v2
import cv2

from sklearn.model_selection import StratifiedKFold 

import timm

warnings.filterwarnings("ignore")

In [7]:
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

In [8]:
device = torch.device("mps"
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print(f"Device: '{device}'")

Device: 'mps'


In [9]:
SEED = 2026
N_SPLITS = 5
BATCH_SIZE = 64
EPOCHS = 25
LR = 1e-3

In [10]:
def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic=(True)
    torch.backends.cudnn.benchmark=(False)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

In [11]:
class RSNADataset(Dataset):

    def __init__(self, df, transform=None, is_train=True):
        self.df = df 
        self.transform = transform
        self.is_train = is_train 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["image_path"]
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        image = image.astype(np.float32) / 255.0 
        image = np.transpose(image, (2, 0, 1))
        image = torch.tensor(image, dtype=torch.float32)

        if self.is_train:
            label = torch.tensor(row["label"], dtype=torch.float32)
            return image, label 

        return image 